# 🎯 YOLOv8 + DeepSORT Object Tracker
**Run each cell top-to-bottom.**  
Upload your video in Cell 3 → the output video downloads automatically at the end.

> ⚡ **Tip:** Go to `Runtime → Change runtime type → T4 GPU` for much faster processing.

## Cell 1 — Install dependencies

In [ ]:
!pip install -q ultralytics deep-sort-realtime opencv-python-headless
print('✅ All packages installed.')

## Cell 2 — Imports

In [ ]:
import cv2
import numpy as np
import time
import os
from pathlib import Path
from IPython.display import display, HTML
from google.colab import files
from ultralytics import YOLO
from deep_sort_realtime.deepsort_tracker import DeepSort

print('✅ Imports OK')

## Cell 3 — Upload your video
Click the file-picker and select your input video.

In [ ]:
uploaded = files.upload()   # opens a file-picker dialog

if uploaded:
    VIDEO_PATH = list(uploaded.keys())[0]
    print(f'✅ Uploaded: {VIDEO_PATH}')
else:
    raise ValueError('No file uploaded. Please re-run this cell and select a video.')

## Cell 4 — Configuration
Edit these variables to suit your use-case.

In [ ]:
# ── Model ─────────────────────────────────────────────────────────────────────
MODEL_NAME  = 'yolov8n.pt'    # n=fastest, s, m, l, x=most accurate

# ── Detection ─────────────────────────────────────────────────────────────────
CONF_THRESH = 0.40            # confidence threshold (0-1)

# ── Output ────────────────────────────────────────────────────────────────────
os.makedirs('output', exist_ok=True)
OUTPUT_PATH = 'output/tracked.mp4'

print('✅ Config set')
print(f'   Input  → {VIDEO_PATH}')
print(f'   Model  → {MODEL_NAME}')
print(f'   Output → {OUTPUT_PATH}')

## Cell 5 — Helper functions

In [ ]:
def draw_fps(frame, fps):
    txt = f'FPS: {fps:.1f}'
    (tw, th), _ = cv2.getTextSize(txt, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)
    x = frame.shape[1] - tw - 12
    cv2.rectangle(frame, (x-4, 6), (x+tw+4, th+14), (0, 0, 0), -1)
    cv2.putText(frame, txt, (x, th+10),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 180), 2, cv2.LINE_AA)


def draw_counter(frame, label, count, y_off):
    txt = f'{label}: {count}'
    cv2.rectangle(frame, (8, y_off-2), (210, y_off+22), (0, 0, 0), -1)
    cv2.putText(frame, txt, (12, y_off+16),
                cv2.FONT_HERSHEY_SIMPLEX, 0.65, (255, 220, 50), 2, cv2.LINE_AA)


print('✅ Helper functions defined')

## Cell 6 — Load models

In [ ]:
print(f'Loading YOLO model: {MODEL_NAME} …')
model = YOLO(MODEL_NAME)          # auto-downloads on first run

print('Initialising DeepSORT tracker …')
tracker = DeepSort(
    max_age=30,
    n_init=3,
    nms_max_overlap=1.0,
    max_cosine_distance=0.3,
    nn_budget=None,
    embedder='mobilenet',
    half=True,
    bgr=True,
)

coco_names = model.names
print('✅ Models ready')

## Cell 7 — Run the tracker 🚀

In [ ]:
# ── Open video ────────────────────────────────────────────────────────────────
cap = cv2.VideoCapture(VIDEO_PATH)
if not cap.isOpened():
    raise FileNotFoundError(f'Cannot open video: {VIDEO_PATH}')

W     = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
H     = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
FPS   = cap.get(cv2.CAP_PROP_FPS) or 25.0
TOTAL = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
print(f'Input  : {W}×{H} @ {FPS:.1f}fps  ({TOTAL} frames)')

# ── VideoWriter ───────────────────────────────────────────────────────────────
writer = cv2.VideoWriter(
    OUTPUT_PATH,
    cv2.VideoWriter_fourcc(*'mp4v'),
    FPS, (W, H)
)

# ── Colour palette (one colour per track ID) ──────────────────────────────────
rng     = np.random.default_rng(42)
palette = rng.integers(80, 255, size=(512, 3)).tolist()

# ── Main loop ─────────────────────────────────────────────────────────────────
frame_idx = 0
t_prev    = time.perf_counter()

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # 1️⃣  YOLO detection
    results = model(frame, conf=CONF_THRESH, verbose=False)[0]

    ds_inputs = []
    for box in results.boxes:
        cls_id        = int(box.cls[0])
        conf          = float(box.conf[0])
        x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
        ds_inputs.append(([x1, y1, x2-x1, y2-y1], conf, cls_id))

    # 2️⃣  DeepSORT update
    tracks = tracker.update_tracks(ds_inputs, frame=frame)

    # 3️⃣  Draw tracks
    counts = {}
    for track in tracks:
        if not track.is_confirmed():
            continue
        tid    = track.track_id
        ltrb   = track.to_ltrb()
        x1, y1, x2, y2 = map(int, ltrb)
        cls_id = track.get_det_class()
        label  = coco_names.get(cls_id, '?') if cls_id is not None else '?'
        counts[label] = counts.get(label, 0) + 1

        color = tuple(palette[int(tid) % len(palette)])
        cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)

        tag = f'#{tid} {label}'
        (tw, th), _ = cv2.getTextSize(tag, cv2.FONT_HERSHEY_SIMPLEX, 0.55, 2)
        cv2.rectangle(frame, (x1, y1-th-8), (x1+tw+6, y1), color, -1)
        cv2.putText(frame, tag, (x1+3, y1-4),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 2, cv2.LINE_AA)

    # 4️⃣  HUD: FPS + per-class counters
    t_now   = time.perf_counter()
    fps_now = 1.0 / max(t_now - t_prev, 1e-6)
    t_prev  = t_now
    draw_fps(frame, fps_now)

    for i, (cls_name, cnt) in enumerate(counts.items()):
        draw_counter(frame, cls_name, cnt, y_off=10 + i*28)

    # 5️⃣  Write frame
    writer.write(frame)
    frame_idx += 1

    if frame_idx % 30 == 0:
        pct = (frame_idx / TOTAL * 100) if TOTAL > 0 else 0
        print(f'  … {frame_idx}/{TOTAL} frames ({pct:.0f}%)')

cap.release()
writer.release()
print(f'\n✅ Done! {frame_idx} frames → {OUTPUT_PATH}')

## Cell 8 — Re-encode with H.264 for browser playback
The raw `mp4v` output isn't playable in most browsers. This cell re-encodes it with H.264.

In [ ]:
H264_PATH = OUTPUT_PATH.replace('.mp4', '_h264.mp4')
!ffmpeg -y -loglevel error -i {OUTPUT_PATH} -vcodec libx264 -crf 23 -pix_fmt yuv420p {H264_PATH}
print(f'✅ H.264 video ready: {H264_PATH}')

## Cell 9 — Preview in notebook

In [ ]:
import base64

with open(H264_PATH, 'rb') as f:
    video_b64 = base64.b64encode(f.read()).decode()

display(HTML(f'''
<video width="720" controls>
  <source src="data:video/mp4;base64,{video_b64}" type="video/mp4">
  Your browser does not support the video tag.
</video>
'''))

## Cell 10 — Download the output video ⬇️

In [ ]:
files.download(H264_PATH)
print(f'⬇️  Download triggered for: {H264_PATH}')